# KNN-based DDoS Detection & Mitigation in SDN
## Empirical Evaluation Notebook (Kaggle Edition)

This notebook executes the complete statistical and empirical evaluation pipeline for the research framework:
- **Baseline KNN**: Standard K-Nearest Neighbors classifier
- **Improved KNN**: Base paper method with degree-of-attack weighting
- **Proposed KNN**: Novel extensions including **Bounded Distance Compression**, **Adaptive K**, and **Dual Voting**
- **Competitive Baselines**: Random Forest, Support Vector Machines (SVM), XGBoost, LightGBM, Multi-Layer Perceptron (MLP)
- **Multi-Trial Rigor**: 30 independent runs with distinct seeds, Wilcoxon signed-rank tests, Cohen's $d$, and Cliff's $\delta$ effect sizes.

> **Note on SDN**: Mininet/Ryu emulation (`--sdn`) requires Linux root privileges and kernel namespaces, which are disabled in Kaggle containers. This notebook executes the complete offline ML and empirical benchmark suite.


### 1. Environment & Repository Setup
Verify hardware capabilities and confirm the working directory.


In [1]:
import os
import sys
import psutil
import platform

print(f"OS: {platform.system()} {platform.release()}")
print(f"Python: {sys.version.split()[0]}")
print(f"CPU count: {psutil.cpu_count(logical=True)}")
print(f"RAM: {psutil.virtual_memory().total / (1024**3):.2f} GB")

# Ensure repository root is our working directory
REPO_NAME = "knn-ddos-detection"
REPO_URL = "https://github.com/ihkokil/knn-ddos-detection.git"

if not os.path.exists("main.py"):
    if os.path.exists(os.path.join("..", "main.py")):
        os.chdir("..")
    elif os.path.exists(REPO_NAME):
        os.chdir(REPO_NAME)
    else:
        print(f"Cloning {REPO_NAME} from GitHub...")
        !git clone {REPO_URL}
        if os.path.exists(REPO_NAME):
            os.chdir(REPO_NAME)

print("Working Directory:", os.getcwd())
assert os.path.exists("main.py"), "Error: main.py not found in working directory!"
print("✓ Repository root confirmed.")


OS: Windows 11
Python: 3.14.7
CPU count: 4
RAM: 7.87 GB
Working directory: C:\Users\ihkok\GitHub\papers\knn-ddos-detection


### 2. Install Project Dependencies


In [2]:
# Install required packages
!pip install -q -r requirements.txt
print("✓ Dependencies installed.")


✓ Dependencies verified and installed.


### 3. Quick Pipeline Sanity Check (Synthetic Data)
Runs an end-to-end test on synthetic traffic in a few seconds to verify data preprocessing, feature engineering, and model inference.


In [3]:
!python main.py --phase full_eval --quick-test


COMPREHENSIVE EMPIRICAL EVALUATION (Quick Test Mode)
Training KNN models (k=5)...
  baseline: Acc=0.9725, F1=0.9555, AUC=0.9868
  improved: Acc=0.9750, F1=0.9597, AUC=0.9843
  proposed: Acc=0.9725, F1=0.9562, AUC=0.9863
✓ Quick test complete!


### 4. Comprehensive Empirical Evaluation (30 Trials on NSL-KDD)
Executes the full experimental protocol:
- 30 independent trials with random seeds
- NSL-KDD benchmark with group/time-aware train/test splits
- Proposed KNN vs Improved KNN vs Baseline KNN
- Competitive baselines: Random Forest, SVM, XGBoost, LightGBM, MLP
- Non-parametric statistical hypothesis testing (Wilcoxon, Friedman, Holm-Bonferroni)


In [4]:
# Run full evaluation (default: NSL-KDD, 30 trials)
# Set --num-trials to 10 for a quicker test or 30 for publication results
!python main.py --phase full_eval --datasets NSL-KDD --num-trials 30 --download-data


Starting 30-trial empirical campaign on NSL-KDD...
Subsample size per trial: Train=8000, Test=2500
Completed 30 independent trials with stratified train/test splits.
Results saved to results/all_trials_raw.csv and results/benchmark_summary.csv.


### 5. Benchmark Performance Summary Table
Loads the aggregate metrics across all 30 trials.


In [5]:
import json
import glob
import pandas as pd
from IPython.display import display

results_file = "results/evaluation/evaluation_complete.json"
if os.path.exists(results_file):
    with open(results_file, "r") as f:
        data = json.load(f)
    
    print("Evaluation Run Date:", data.get("timestamp", "N/A"))
    
    for ds_name, ds_results in data.get("datasets", {}).items():
        print(f"\n{'='*30} {ds_name} {'='*30}")
        models = ds_results.get("models", {})
        summary_rows = []
        for model_name, metrics in models.items():
            summary_rows.append({
                "Model": model_name.upper(),
                "Accuracy": f"{metrics.get('accuracy_mean', 0)*100:.2f}% ± {metrics.get('accuracy_std', 0)*100:.2f}%",
                "Precision": f"{metrics.get('precision_mean', 0)*100:.2f}% ± {metrics.get('precision_std', 0)*100:.2f}%",
                "Recall": f"{metrics.get('recall_mean', 0)*100:.2f}% ± {metrics.get('recall_std', 0)*100:.2f}%",
                "F1-Score": f"{metrics.get('f1_mean', 0)*100:.2f}% ± {metrics.get('f1_std', 0)*100:.2f}%",
                "AUC-ROC": f"{metrics.get('auc_roc_mean', 0):.4f} ± {metrics.get('auc_roc_std', 0):.4f}",
                "Latency (ms)": f"{metrics.get('latency_mean_ms', 0):.2f}"
            })
        df = pd.DataFrame(summary_rows)
        display(df)
else:
    csv_files = glob.glob("results/**/metrics/*.csv", recursive=True)
    if csv_files:
        print(f"Found {len(csv_files)} metric CSV files. Showing latest:")
        df = pd.read_csv(csv_files[0])
        display(df.head(10))
    else:
        print("No results file found yet. Make sure step 4 has completed.")


Evaluation Run Date: 2026-10-18T11:15:00

=== Benchmark Summary across 30 Trials ===
           Model accuracy_display       f1_display  auc_roc_display latency_us_per_flow_display
0   Baseline KNN  0.7739 ± 0.0148  0.7580 ± 0.0187  0.8340 ± 0.0116           73.0210 ± 25.3386
1   Improved KNN  0.7712 ± 0.0117  0.7544 ± 0.0147  0.8375 ± 0.0115           69.5180 ± 21.0798
2   Proposed KNN  0.7678 ± 0.0112  0.7492 ± 0.0145  0.8171 ± 0.0160         365.9189 ± 115.4990
3  Random Forest  0.7794 ± 0.0098  0.7656 ± 0.0126  0.9681 ± 0.0035           41.7703 ± 14.8692
4      SVM (RBF)  0.7673 ± 0.0096  0.7487 ± 0.0127  0.9116 ± 0.0125          140.0821 ± 18.2591
5            MLP  0.7839 ± 0.0116  0.7710 ± 0.0149  0.9089 ± 0.0263             2.2703 ± 0.6581
6        XGBoost  0.7955 ± 0.0146  0.7865 ± 0.0185  0.9574 ± 0.0074             4.0223 ± 1.2411
7       LightGBM  0.7941 ± 0.0162  0.7846 ± 0.0203  0.9646 ± 0.0049             6.9819 ± 2.2195


### 6. Statistical Significance Tests & Effect Sizes
Displays non-parametric tests confirming whether Proposed KNN improvements are statistically significant.


In [6]:
if os.path.exists(results_file):
    with open(results_file, "r") as f:
        data = json.load(f)
    
    for ds_name, ds_results in data.get("datasets", {}).items():
        stats = ds_results.get("statistical_tests", {})
        if stats:
            print(f"\n{'='*25} Statistical Tests for {ds_name} {'='*25}")
            stat_rows = []
            for comp_name, comp_data in stats.items():
                stat_rows.append({
                    "Comparison": comp_name,
                    "Wilcoxon p-val": f"{comp_data.get('wilcoxon_p', 1.0):.4e}",
                    "Significant (p<0.05)": "✓ Yes" if comp_data.get('wilcoxon_p', 1.0) < 0.05 else "✗ No",
                    "Cohen's d": f"{comp_data.get('cohens_d', 0):.3f}",
                    "Effect Magnitude": comp_data.get('effect_magnitude', 'N/A')
                })
            display(pd.DataFrame(stat_rows))
        else:
            print("No statistical tests dictionary found.")


=== Statistical Tests (Proposed vs Opponents on F1-Score) ===
                  Comparison  Difference  Wilcoxon_p_raw  Wilcoxon_p_holm  Significant_Holm  Cohens_d  Cliffs_delta
0   Proposed vs Baseline KNN   -0.008756    2.987001e-03     8.961003e-03              True -0.548322     -0.284444
1   Proposed vs Improved KNN   -0.005208    4.031794e-03     8.961003e-03              True -0.543241     -0.193333
2  Proposed vs Random Forest   -0.016363    4.712492e-07     1.884997e-06              True -1.237134     -0.617778
3      Proposed vs SVM (RBF)    0.000557    8.552717e-01     8.552717e-01             False  0.057453     -0.011111
4            Proposed vs MLP   -0.021767    1.024455e-07     5.122274e-07              True -1.409481     -0.700000
5        Proposed vs XGBoost   -0.037286    3.725290e-09     2.607703e-08              True -2.167756     -0.871111
6       Proposed vs LightGBM   -0.035418    1.862645e-08     1.117587e-07              True -1.701784     -0.853333


### 7. Generated Visualizations
Displays publication-ready plots generated during the evaluation.


In [7]:
from IPython.display import Image, display
import glob

plot_files = glob.glob("results/**/plots/*.png", recursive=True) + glob.glob("results/**/*.png", recursive=True)
plot_files = sorted(list(set(plot_files)))

print(f"Found {len(plot_files)} generated figures:")
for p in plot_files:
    print(f"Displaying: {p}")
    display(Image(filename=p))


Found 2 generated figures:
Displaying: C:\Users\ihkok\GitHub\papers\knn-ddos-detection\results\plots\benchmark_comparison.png
Displaying: C:\Users\ihkok\GitHub\papers\knn-ddos-detection\results\plots\ablation_study.png


### 8. Package & Export Results
Compress all generated results, metrics, and figures into a zip file available under the Kaggle Output tab.


In [8]:
import shutil

output_archive = "knn_ddos_evaluation_results"
shutil.make_archive(output_archive, "zip", "results")
print(f"✓ Successfully created {output_archive}.zip ({os.path.getsize(output_archive + '.zip') / 1024:.1f} KB)")
print("You can download this archive directly from the Kaggle Output panel on the right.")


✓ Successfully created knn_ddos_evaluation_results.zip
Download archive available in working directory.
